# 第一部分：僵化的祖先 (可视化增强版)
### *GlobalCorp 资深分析师 Alex 的职业高光与危机*

**Alex** 凭借一套精美的自动化报表在 GlobalCorp 站稳了脚跟。每当 `Standard_Sales.csv` 导入时，系统不仅能计算出准确的税务，还能生成让老板眼前一亮的图表。

在这一章中，我们将：
1. **构建企业级仪表盘**：为标准数据生成漂亮的收入分布图。
2. **硬编码可视化逻辑**：展示当绘图逻辑也变得“僵化”时会发生什么。
3. **体验从“高光”到“事故”**：当新数据导致可视化引擎崩溃时，Alex 该如何向老板交代？

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # suppress all warnings
import matplotlib
matplotlib.font_manager._log.setLevel(60)  # suppress findfont warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 设置企业级绘图风格
sns.set_theme(style="whitegrid")
# 使用 macOS 内置中文字体
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

os.makedirs("enterprise_data", exist_ok=True)
print("可视化环境及数据目录准备就绪。")


## 1. 复杂企业数据生成
我们保留之前的多维数据结构，确保包含：地区、产品类别、单价、折扣等。

In [ ]:
# --- 1.1 标准销售导出 (Legacy) ---
legacy_data = {
    'TransactionID': ['TXN_001', 'TXN_002', 'TXN_003', 'TXN_004', 'TXN_005', 'TXN_006'],
    'Date': ['2023-12-01', '2023-12-01', '2023-12-02', '2023-12-03', '2023-12-03', '2023-12-04'],
    'Region': ['US', 'EMEA', 'US', 'APAC', 'EMEA', 'APAC'],
    'Product_Category': ['Software', 'Hardware', 'Software', 'Services', 'Hardware', 'Software'],
    'Quantity': [10, 1, 5, 2, 1, 8],
    'Unit_Price': [500.0, 1200.0, 500.0, 2000.0, 1200.0, 500.0],
    'Discount': [50.0, 0.0, 20.0, 100.0, 0.0, 40.0]
}
pd.DataFrame(legacy_data).to_csv("enterprise_data/standard_sales.csv", index=False)

# --- 1.2 现代市场导出 (带有数据漂移) ---
modern_data = {
    'tx_id': ['MKT_999', 'MKT_1000', 'MKT_1001'],
    'market': ['US', 'EMEA', 'LATAM'],
    'prod_cat': ['Software', 'Hardware', 'Services'],
    'qty': [20, 2, 1],
    'price_per_unit': ['$500.00', '$1,250.00', '$2,100.00'],
    'rebate': [100.0, 10.0, 50.0]
}
pd.DataFrame(modern_data).to_csv("enterprise_data/modern_marketing.csv", index=False)
print("多维企业数据已更新。")

## 2. 确定性分析与可视化引擎

Alex 编写的函数不仅能计算数据，还能生成一份“性感”的报告。

In [ ]:
def globalcorp_visual_report(file_path):
    df = pd.read_csv(file_path)
    
    # --- A. 确定性财务计算 ---
    # 计算公式：(数量 * 单价) - 折扣
    df['Total_Revenue'] = (df['Quantity'] * df['Unit_Price']) - df['Discount']
    
    # --- B. 确定性可视化逻辑 (硬编码绘图) ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. 地区收入分布 (柱状图)
    region_data = df.groupby('Region')['Total_Revenue'].sum().sort_values(ascending=False)
    sns.barplot(x=region_data.index, y=region_data.values, hue=region_data.index, ax=ax1, palette="viridis", legend=False)
    ax1.set_title("按地区汇总的净收入", fontsize=14, fontweight='bold')
    ax1.set_ylabel("收入 (USD)")
    
    # 2. 产品类别贡献 (环形图)
    cat_data = df.groupby('Product_Category')['Total_Revenue'].sum()
    ax2.pie(cat_data, labels=cat_data.index, autopct='%1.1f%%', startangle=140, 
            colors=sns.color_palette("pastel"), wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    # 画一个中心的白圆，做成环形图
    centre_circle = plt.Circle((0,0), 0.70, fc='white')
    ax2.add_artist(centre_circle)
    ax2.set_title("产品类别贡献率", fontsize=14, fontweight='bold')

    plt.suptitle(f"GlobalCorp 业务分析报告: {os.path.basename(file_path)}", fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    return "报告生成成功。"

## 3. 高光时刻：标准数据的完美呈现
运行下面的代码，看看 Alex 赖以生存的“完美报表”。

In [ ]:
try:
    status = globalcorp_visual_report("enterprise_data/standard_sales.csv")
    print(status)
except Exception as e:
    print(f"❌ 失败: {e}")

## 4. 危机降临：当可视化遇到“业务变迁”
现在，老板说：“Alex，那个很酷的报表，帮我也跑一下市场部的新数据。”

In [ ]:
try:
    print("正在尝试为市场部生成报表...")
    globalcorp_visual_report("enterprise_data/modern_marketing.csv")
except Exception as e:
    print("\n❌ 系统崩溃")
    print(f"错误原因: 可视化引擎找不到特定的列名进行计算或绘图。")
    print(f"具体错误: {e}")

## 复盘：不仅仅是数据，连“展示”也是脆弱的

在企业中，**图表是决策的依据**。但通过这个 Notebook 我们发现：
1. **计算与展示是强耦合的**：如果底层的 `Quantity` 变成了 `qty`，不仅财务算不出数，漂亮的环形图也无法渲染。
2. **硬编码的审美代价**：Alex 的代码里写死了 `groupby('Region')`。当市场部使用 `market` 作为地区标识时，绘图引擎会直接报 `KeyError`。

### Alex 的出路？
如果 Alex 继续走“硬编码”的老路，他必须为市场部再写一套几乎一样的绘图逻辑。随着部门越来越多，他的代码库将变成一个无法维护的“屎山”。

**下一章：我们将向 Alex 介绍 LLM 顾问。我们将看到，AI 如何通过理解“Region 就是市场”，在不修改绘图引擎的情况下，让报表重新动起来。**